### preprocessing of datasets 

### 1 importing modules and loading of dataset 


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder


In [2]:
df = pd.read_csv("../Data/dataset.csv")
print(df.shape)
print(df.dtypes)
print(df.describe())
print(df.isnull().sum())
print(df.nunique())
num_cols = ["Age","Height","FCVC","NCP","CH2O","FAF","TUE"]
non_numeric_col = ["Gender","family_history_with_overweight","FAVC","CAEC",
"SMOKE","SCC","CALC","MTRANS","NObeyesdad"]

(2111, 17)
Gender                                str
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight        str
FAVC                                  str
FCVC                              float64
NCP                               float64
CAEC                                  str
SMOKE                                 str
CH2O                              float64
SCC                                   str
FAF                               float64
TUE                               float64
CALC                                  str
MTRANS                                str
NObeyesdad                            str
dtype: object
               Age       Height       Weight         FCVC          NCP  \
count  2111.000000  2111.000000  2111.000000  2111.000000  2111.000000   
mean     24.312600     1.701677    86.586058     2.419043     2.685628   
std       6.345968     0.093305    26.1

### 2 handling missing values sincve there is no missing values we are skiping these step 

### 3 . outlier detection 


In [3]:
def detect_outliers(df,col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    IQR = q3 - q1
    lower = q1 - 1.5 * IQR
    upper = q3 + 1.5 * IQR
    print(f"outlier at col {col} is {len( df[(df[col]<lower)| (df[col]>upper)])}")
    # transform outliers
    df[col] = np.where(df[col] > upper, upper, df[col])
    df[col] = np.where(df[col] < lower, lower, df[col])

for col in num_cols:
    outliers = detect_outliers(df,col)


outlier at col Age is 168
outlier at col Height is 1
outlier at col FCVC is 0
outlier at col NCP is 579
outlier at col CH2O is 0
outlier at col FAF is 0
outlier at col TUE is 0


### 4. encoding data 


In [4]:
encoder = LabelEncoder()

for col in non_numeric_col:
    print(f"{col} has {df[col].nunique()} values in it ")
non_bin_cols = ["CAEC","CALC","MTRANS","NObeyesdad"]

Gender has 2 values in it 
family_history_with_overweight has 2 values in it 
FAVC has 2 values in it 
CAEC has 4 values in it 
SMOKE has 2 values in it 
SCC has 2 values in it 
CALC has 4 values in it 
MTRANS has 5 values in it 
NObeyesdad has 7 values in it 


for the above we can identify the some binary col are gender , FAVC, CAEC , SMOKE , SCC


In [5]:
bin_cols = ["Gender" , "FAVC" , "SMOKE" , "SCC","family_history_with_overweight"]
for col in bin_cols:
    df[col] = encoder.fit_transform(df[col])


### ordinal encoding

In [6]:
for col in non_bin_cols:
    print(f"column {col} has these value {df[col].unique()}")

target_encoding_cols =["CAEC","CALC",]
map_guided = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}

for col in target_encoding_cols:
    df[col] = df[col].map(map_guided)

map_guide_for_NObeyesdad = {
      'Normal_Weight':2,  'Overweight_Level_I':3, 'Overweight_Level_II':4,
      'Obesity_Type_I':5, 'Insufficient_Weight':1,     'Obesity_Type_II':6,
    'Obesity_Type_III':7    
}

df["NObeyesdad"] = df["NObeyesdad"].map(map_guide_for_NObeyesdad)



column CAEC has these value <StringArray>
['Sometimes', 'Frequently', 'Always', 'no']
Length: 4, dtype: str
column CALC has these value <StringArray>
['no', 'Sometimes', 'Frequently', 'Always']
Length: 4, dtype: str
column MTRANS has these value <StringArray>
['Public_Transportation', 'Walking', 'Automobile', 'Motorbike', 'Bike']
Length: 5, dtype: str
column NObeyesdad has these value <StringArray>
[      'Normal_Weight',  'Overweight_Level_I', 'Overweight_Level_II',
      'Obesity_Type_I', 'Insufficient_Weight',     'Obesity_Type_II',
    'Obesity_Type_III']
Length: 7, dtype: str


### one hot enconding 

In [7]:
encoder = OneHotEncoder()
encoded = encoder.fit_transform(df[["MTRANS"]]).toarray()
encoder_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())
df = df.drop(columns=["MTRANS"])
df = pd.concat([df, encoder_df], axis=1)

### DOMAIN SPECIFIC TERMS 


In [8]:

# introducing squared terms to capture non linear relationship 
df['Height_squared'] = df['Height'] ** 2
df['Age_squared'] = df['Age'] ** 2
df['CH2O_squared'] = df['CH2O'] ** 2  
## DOMAIN SPECIFIC TERMS
df['Height_x_FAF'] = df['Height'] * df['FAF'] 
df['Age_x_CH2O'] = df['Age'] * df['CH2O']
df['Genetics_x_FAVC'] = df['family_history_with_overweight'] * df['FAVC']

In [9]:
df.info()
df["Age"] = df["Age"].astype(int)
df['NCP']= df['NCP'].round()
df.drop("NObeyesdad",axis=1,inplace=True)

<class 'pandas.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          2111 non-null   int64  
 1   Age                             2111 non-null   float64
 2   Height                          2111 non-null   float64
 3   Weight                          2111 non-null   float64
 4   family_history_with_overweight  2111 non-null   int64  
 5   FAVC                            2111 non-null   int64  
 6   FCVC                            2111 non-null   float64
 7   NCP                             2111 non-null   float64
 8   CAEC                            2111 non-null   int64  
 9   SMOKE                           2111 non-null   int64  
 10  CH2O                            2111 non-null   float64
 11  SCC                             2111 non-null   int64  
 12  FAF                             2111 non-null

In [10]:
int_columns = df.select_dtypes(include=['int64']).columns
df[int_columns] = df[int_columns].astype(float)
df.to_csv('processed.csv', index=False)